# Re-run: reproductions of published numbers (RQ1)

The per-case outputs (`per-case-outputs.jsonl`, published with the release) hold one `gate` record per
reproduction, with the benchmark's value, the published target and, for the pyKT models, pyKT's own loop.
This notebook recomputes every number of Tables 2 and 3 from those records and checks it against the article.
It needs only the standard library.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "knowledge-tracing-benchmark-2026"   # set when the report is built
DATA_NAME = 'per-case-outputs.jsonl'     # this release's per-case file, set when the report is built (None: it has none)


def find_release_file(name=None):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    name = name or DATA_NAME
    if not name:
        raise FileNotFoundError(f"Release {RELEASE_ID} publishes no per-case file; this notebook reads its "
                                f"code package (code_dir).")
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        if (d / name).is_file():
            return d / name
        gz = d / (name + ".gz")
        if gz.is_file():
            # The report offers the data file gzipped; unpack it once beside the download, so a notebook that
            # opens the file as text reads the same bytes either way.
            try:
                (d / name).write_bytes(gzip.decompress(gz.read_bytes()))
                return d / name
            except OSError:
                return gz
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def is_this_release(d):
    """True when folder d holds this release, not another one's code/ found higher up: d is named after the
    release, or its code-manifest.txt or a release file (*.yaml) names it."""
    if d.name == RELEASE_ID:
        return True
    texts = [d / "code-manifest.txt", *sorted(d.glob("*.yaml"))]
    return any(f.is_file() and RELEASE_ID in f.read_text(encoding="utf-8", errors="ignore") for f in texts)


def code_dir():
    """The release's code package (its code/ folder), looked for only by a notebook that reads it: in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file() and is_this_release(d):
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-outputs.jsonl


In [2]:
recs = [json.loads(x) for x in open(find_release_file("per-case-outputs.jsonl"), encoding="utf-8")]
gates = [r for r in recs if r["record"] == "gate" and r["attempt"] == "counted"]
first = [r for r in recs if r["record"] == "gate" and r["attempt"] != "counted"]
f4 = lambda x: f"{x:.4f}".replace("-", "−")
s4 = lambda x: f"{x:+.4f}".replace("-", "−")

# Table 2: Gervet et al. (2020) targets, band 0.010
want2 = {("IRT", "gervet"): ("0.7432", "−0.0058", "PASS"),
         ("PFA", "gervet"): ("0.7571", "+0.0161", "HARNESS_FAIL"),
         ("PFA", "gervet-amended"): ("0.7571", "−0.0039", "PASS"),
         ("Best-LR", "gervet"): ("0.8046", "+0.0016", "PASS"),
         ("DKT", "gervet"): ("0.7980", "+0.0080", "PASS")}
for g in gates:
    if g["kind"].startswith("gervet"):
        diff = g["bench_auc"] - g["target"]
        reading = "PASS" if abs(diff) <= 0.010 else "HARNESS_FAIL"
        got = (f4(g["bench_auc"]), s4(diff), reading)
        print(g["model"], g["kind"], g["target"], got)
        assert got == want2[(g["model"], g["kind"])]
print("Table 2 matches the article.")

IRT gervet 0.749 ('0.7432', '−0.0058', 'PASS')
PFA gervet 0.741 ('0.7571', '+0.0161', 'HARNESS_FAIL')
PFA gervet-amended 0.761 ('0.7571', '−0.0039', 'PASS')
Best-LR gervet 0.803 ('0.8046', '+0.0016', 'PASS')
DKT gervet 0.79 ('0.7980', '+0.0080', 'PASS')
Table 2 matches the article.


In [3]:
# Table 3: pyKT gate, reproduction band 0.010 and loop band 0.005
want3 = {"AKT": ("0.8136", "−0.0072", "+0.0030"), "simpleKT": ("0.8111", "−0.0049", "+0.0026"),
         "IEKT": ("0.8052", "−0.0073", "−0.0033"), "HawkesKT": ("0.8028", "−0.0040", "−0.0038"),
         "LPKT": ("0.7980", "−0.0076", "+0.0016")}
for g in gates:
    if g["kind"] == "pykt":
        rep = g["pykt_primary_auc"] - g["target"]
        loop = g["bench_auc"] - g["pykt_loop_auc"]
        assert abs(rep - g["repro_diff"]) < 1e-12 and abs(loop - g["loop_diff"]) < 1e-12
        ok = abs(rep) <= 0.010 and abs(loop) <= 0.005
        got = (f4(g["pykt_primary_auc"]), s4(rep), s4(loop))
        print(g["model"], g["target"], got, "PASS" if ok else "FAIL")
        assert got == want3[g["model"]] and ok and g["reading"] == "PASS"
print("Table 3 matches the article.")

AKT 0.8208 ('0.8136', '−0.0072', '+0.0030') PASS
simpleKT 0.816 ('0.8111', '−0.0049', '+0.0026') PASS
IEKT 0.8125 ('0.8052', '−0.0073', '−0.0033') PASS
HawkesKT 0.8068 ('0.8028', '−0.0040', '−0.0038') PASS
LPKT 0.8056 ('0.7980', '−0.0076', '+0.0016') PASS
Table 3 matches the article.


In [4]:
# The two failed first attempts the text cites: HawkesKT's loop 0.157 below pyKT's, LPKT's reproduction -0.0156
fa = {g["model"]: g for g in first}
assert f"{-fa['HawkesKT']['loop_diff']:.3f}" == "0.157" and fa["HawkesKT"]["reading"] == "HARNESS_FAIL"
assert s4(fa["LPKT"]["repro_diff"]) == "−0.0156" and fa["LPKT"]["reading"] == "HARNESS_FAIL"
print("First attempts as printed: HawkesKT loop -0.157, LPKT reproduction -0.0156.")

First attempts as printed: HawkesKT loop -0.157, LPKT reproduction -0.0156.
